In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C_dec'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [12]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [13]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 106)

In [14]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [15]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(10, 106)

array([[1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       ...,
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 0.5, 0.5, 0.5]])

## Run models

In [17]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [18]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
trh = 0.0001
display((abs(a_coeffs_df) >trh).sum(axis = "columns"))
display(a_coeffs_df)

CDK1_2      14
CDK4_6       6
EGFR        22
Estrogen    10
FGFR         9
PI3K        15
p53         27
TOP2A       11
Src          5
SMAD3        1
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
CDK1_2,-4.744761e-07,3.960327e-08,-1.092155e-05,1.895114e-05,0.000007,0.000007,0.000005,-2.389898e-05,-0.000003,1.318738e-05,...,0.000013,-9.485123e-08,-0.000003,0.000007,-4.243907e-06,0.000016,0.000007,0.000016,1.369052e-05,0.000014
CDK4_6,1.016938e-05,-1.443022e-05,1.617582e-07,2.201106e-05,0.000011,0.000030,0.000007,-1.638332e-05,-0.000002,-2.773810e-05,...,0.000009,-1.479752e-05,0.000011,-0.000016,-3.327698e-06,-0.000060,0.000013,0.000015,-3.371045e-05,-0.000020
EGFR,-4.287425e-06,1.603530e-05,-1.559162e-05,-1.296990e-05,-0.000017,-0.000300,0.000012,1.417961e-05,-0.000013,-2.922756e-02,...,-0.000041,-1.664773e-05,-0.000028,-0.000011,-2.920556e-06,0.000151,-0.000010,0.000040,2.834185e-05,-0.000011
Estrogen,-1.534138e-05,-1.290273e-05,-5.204225e-06,3.768757e-07,0.000001,-0.000036,-0.000027,6.056299e-06,0.000027,-2.737052e-01,...,0.000003,-9.837718e-06,-0.000017,0.000014,-9.266168e-06,-0.000025,0.000016,0.000014,1.557417e-07,0.000011
FGFR,-3.880220e-04,1.867959e-05,-1.987765e-05,1.592055e-05,0.000011,-0.000007,0.000014,-1.911629e-05,0.000004,-4.159447e-05,...,-0.000004,-7.578389e-07,0.000019,-0.000004,1.153632e-05,0.000022,0.000014,0.000013,-8.071577e-06,0.000048
PI3K,5.677687e-06,1.796879e-05,-1.816571e-06,1.608590e-07,-0.000006,0.000002,0.000006,-7.509435e-06,0.000004,-2.867144e-06,...,-0.000005,2.042256e-05,0.000014,-0.000003,-4.148821e-07,-0.000031,0.000010,-0.000003,-8.833958e-06,0.000012
p53,1.004210e-06,9.176720e-06,-2.759556e-06,-1.598436e-05,0.000016,0.000009,0.258393,6.907094e-07,-0.000011,-1.905830e-05,...,-0.000002,4.029698e-06,-0.000018,0.000023,1.676320e-05,-0.000014,0.000006,0.000029,2.524541e-05,0.000020
TOP2A,-6.317954e-06,1.682447e-05,-5.944175e-06,9.166762e-06,-0.000002,0.000013,0.000011,-2.041824e-05,0.000013,-1.580663e-08,...,0.000009,2.487233e-05,0.000005,0.000004,-8.656448e-06,0.000010,-0.000006,0.000001,7.903517e-06,0.000002
Src,5.896325e-06,1.480404e-05,1.284042e-05,-6.346685e-07,-0.000003,0.000007,0.000016,-1.552464e-05,0.000015,-4.119466e-05,...,0.000012,-3.151458e-05,-0.000006,-0.000009,1.036616e-05,-0.000014,-0.000024,0.000010,1.113364e-05,0.000004
SMAD3,-4.973374e-05,5.613387e-06,-1.408021e-05,-1.473920e-05,-0.000032,-0.000023,0.000018,1.073517e-05,0.000011,-1.314664e-05,...,-0.000016,-1.840610e-06,0.000019,-0.000012,-3.227561e-05,0.000003,-0.000020,0.000005,9.163573e-07,0.000003


In [19]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [20]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.728809,-0.594966,0.078836,-0.081689,0.094967,-0.285751,0.025600,-0.292817,0.120878,0.058646,...,-0.477712,-0.435273,-0.398574,-0.415532,-0.384505,-0.400906,-0.392687,-0.435683,-0.433020,-0.470333
CDK4_6,-0.253419,-0.435887,-0.320664,-0.199581,-0.529572,-0.105917,0.071964,-0.005714,-0.231109,-0.032318,...,0.192772,0.174359,0.182195,0.119491,0.133592,0.145679,0.162411,0.159861,0.176947,0.148816
EGFR,0.560826,0.461896,0.215374,0.337317,0.440541,0.069678,-0.374897,0.221054,0.270954,-0.053595,...,-1.168748,-0.806060,-0.783816,-0.701253,-0.665495,-0.684721,-0.731162,-0.873795,-0.857880,-0.915029
Estrogen,-0.126450,-0.211202,-0.212511,-0.409488,-0.946806,-0.308706,-0.085617,-0.241987,-0.166795,-0.042034,...,-0.295623,-0.264868,-0.238049,-0.262339,-0.242920,-0.251208,-0.228984,-0.275499,-0.259331,-0.303315
FGFR,-0.067786,-0.197355,-0.077613,0.055243,-0.035806,-0.358686,-0.013911,-0.133003,-0.093474,-0.255184,...,-0.818299,-0.681987,-0.629570,-0.627170,-0.564531,-0.611042,-0.620822,-0.688550,-0.703922,-0.757253
PI3K,-1.939288,-1.727130,-1.501679,-1.309667,-0.691259,-0.196006,-0.027224,-0.490153,-1.185209,-0.216757,...,-0.153804,-0.110577,-0.080306,-0.095372,-0.073719,-0.113371,-0.096138,-0.112758,-0.123783,-0.171081
p53,-0.282725,-0.310014,-0.186759,-0.390680,0.031155,-1.627989,-1.476111,-0.115811,-0.093941,-1.327713,...,0.064832,0.040916,0.053008,0.054518,0.027011,0.059495,0.078142,0.040343,0.079447,0.053808
TOP2A,-0.230390,0.072076,-0.234480,-0.174332,-0.127842,0.049964,0.068540,-1.994020,-0.207025,-0.291724,...,0.121078,0.105863,0.097925,0.082779,0.077295,0.092613,0.098756,0.097988,0.110276,0.110107
Src,-0.920943,-1.663954,0.525676,-1.209609,0.573217,-1.116668,0.495558,0.400356,0.398413,0.449117,...,-0.114102,-0.140965,-0.085586,-0.115775,-0.045394,-0.105658,-0.086543,-0.101180,-0.142361,-0.153170
SMAD3,0.017997,-0.096668,0.026525,-0.283670,0.048734,0.068263,0.022095,0.115961,0.118714,0.052089,...,-0.703939,-0.647946,-0.620606,-0.642048,-0.616811,-0.624165,-0.613684,-0.661611,-0.650213,-0.689409


In [21]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.728809,-0.594966,0.078836,-0.081689,0.094967,-0.285751,0.025600,-0.292817,0.120878,0.058646,...,-0.477712,-0.435273,-0.398574,-0.415532,-0.384505,-0.400906,-0.392687,-0.435683,-0.433020,-0.470333
CDK4_6,-0.253419,-0.435887,-0.320664,-0.199581,-0.529572,-0.105917,0.071964,-0.005714,-0.231109,-0.032318,...,0.192772,0.174359,0.182195,0.119491,0.133592,0.145679,0.162411,0.159861,0.176947,0.148816
EGFR,0.560826,0.461896,0.215374,0.337317,0.440541,0.069678,-0.374897,0.221054,0.270954,-0.053595,...,-1.168748,-0.806060,-0.783816,-0.701253,-0.665495,-0.684721,-0.731162,-0.873795,-0.857880,-0.915029
Estrogen,-0.126450,-0.211202,-0.212511,-0.409488,-0.946806,-0.308706,-0.085617,-0.241987,-0.166795,-0.042034,...,-0.295623,-0.264868,-0.238049,-0.262339,-0.242920,-0.251208,-0.228984,-0.275499,-0.259331,-0.303315
FGFR,-0.067786,-0.197355,-0.077613,0.055243,-0.035806,-0.358686,-0.013911,-0.133003,-0.093474,-0.255184,...,-0.818299,-0.681987,-0.629570,-0.627170,-0.564531,-0.611042,-0.620822,-0.688550,-0.703922,-0.757253
PI3K,-1.939288,-1.727130,-1.501679,-1.309667,-0.691259,-0.196006,-0.027224,-0.490153,-1.185209,-0.216757,...,-0.153804,-0.110577,-0.080306,-0.095372,-0.073719,-0.113371,-0.096138,-0.112758,-0.123783,-0.171081
p53,-0.282725,-0.310014,-0.186759,-0.390680,0.031155,-1.627989,-1.476111,-0.115811,-0.093941,-1.327713,...,0.064832,0.040916,0.053008,0.054518,0.027011,0.059495,0.078142,0.040343,0.079447,0.053808
TOP2A,-0.230390,0.072076,-0.234480,-0.174332,-0.127842,0.049964,0.068540,-1.994020,-0.207025,-0.291724,...,0.121078,0.105863,0.097925,0.082779,0.077295,0.092613,0.098756,0.097988,0.110276,0.110107
Src,-0.920943,-1.663954,0.525676,-1.209609,0.573217,-1.116668,0.495558,0.400356,0.398413,0.449117,...,-0.114102,-0.140965,-0.085586,-0.115775,-0.045394,-0.105658,-0.086543,-0.101180,-0.142361,-0.153170
SMAD3,0.017997,-0.096668,0.026525,-0.283670,0.048734,0.068263,0.022095,0.115961,0.118714,0.052089,...,-0.703939,-0.647946,-0.620606,-0.642048,-0.616811,-0.624165,-0.613684,-0.661611,-0.650213,-0.689409
